In [6]:
import sys
import os

python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path
from pathlib import Path

from pyspark.sql import SparkSession

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "fraud_streaming" / "spark" / "src" / "data_processing.py").is_file()
    ),
    Path.cwd(),
)
spark_dir = project_root / "fraud_streaming" / "spark"
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(spark_dir))

from src.data_processing import compute_window_analytics
from fraud_streaming.redis.config import (
    ANALYTICS_POLL_SECONDS,
    host as redis_host,
    port as redis_port,
)
from fraud_streaming.redis.redis_integration import RedisAnalyticsIntegration

spark = SparkSession.builder.appName("RedisFeatureWindowAnalysis").getOrCreate()
analytics_store = RedisAnalyticsIntegration(host=redis_host, port=redis_port)
analysis_checkpoint_path = (
    project_root / "data" / "checkpoints" / "feature_analysis"
)

In [7]:
def process_feature_analysis_batch(_timer_batch, batch_id):
    transactions = analytics_store.read_feature_store()
    metrics = compute_window_analytics(spark, transactions)
    analytics_store.write_analytics_to_redis(metrics)
    print(f"Updated Redis analytics for batch {batch_id}: {metrics}")

In [8]:
timer_stream = (
    spark.readStream.format("rate")
    .option("rowsPerSecond", 1)
    .load()
)

analysis_query = (
    timer_stream.writeStream
    .foreachBatch(process_feature_analysis_batch)
    .option("checkpointLocation", analysis_checkpoint_path.as_posix())
    .trigger(processingTime=f"{ANALYTICS_POLL_SECONDS} seconds")
    .start()
)
print(f"Started Redis feature analysis; polling every {ANALYTICS_POLL_SECONDS}s")

Started Redis feature analysis; polling every 5s


In [9]:
try:
    analysis_query.awaitTermination()
finally:
    analysis_query.stop()
    analytics_store.close()

Updated Redis analytics for batch 0: {'transaction_count': 6, 'total_amount': 234.97, 'average_amount': 39.16, 'category_counts': {'shopping_pos': 1, 'grocery_pos': 3, 'gas_transport': 1, 'misc_net': 1}, 'window_start': '2026-11-11T09:52:03+00:00', 'window_end': '2026-11-11T09:53:03+00:00', 'updated_at': '2026-11-11T09:53:03+00:00'}
Updated Redis analytics for batch 1: {'transaction_count': 81, 'total_amount': 4832.63, 'average_amount': 59.66, 'category_counts': {'grocery_pos': 3, 'kids_pets': 11, 'grocery_net': 2, 'food_dining': 8, 'gas_transport': 8, 'entertainment': 8, 'misc_pos': 3, 'personal_care': 10, 'health_fitness': 6, 'home': 12, 'misc_net': 2, 'shopping_net': 4, 'shopping_pos': 4}, 'window_start': '2026-11-11T09:53:05+00:00', 'window_end': '2026-11-11T09:54:05+00:00', 'updated_at': '2026-11-11T09:54:05+00:00'}
Updated Redis analytics for batch 2: {'transaction_count': 53, 'total_amount': 2669.41, 'average_amount': 50.37, 'category_counts': {'kids_pets': 6, 'food_dining': 7, 

StreamingQueryException: [STREAM_FAILED] Query [id = 877a6fd8-649b-4b80-ae80-5cf024d43726, runId = c0ba5f5f-d4de-4eee-8437-c85a23b15f46] terminated with exception: [FOREACH_BATCH_USER_FUNCTION_ERROR] An error occurred in the user provided function in foreach batch sink. Reason: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\dev_env\Lib\site-packages\py4j\clientserver.py", line 644, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\dev_env\Lib\site-packages\pyspark\sql\utils.py", line 166, in call
    raise e
  File "c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\dev_env\Lib\site-packages\pyspark\sql\utils.py", line 163, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
    ~~~~